In [3]:
## Building the baseline model what Tagra did

In [4]:
import pandas as pd
import os

DATA_DIR = r"C:\Users\ramku\OneDrive\Documents\Research Practicum\Execution\Reopened bugs dataset"

#loading just 1 project
df = pd.read_csv(os.path.join(DATA_DIR,"joined_project_dataframe_GERONIMO.csv"), low_memory=False)

# y =the answer needed to predict (1=reopened and 0 =not)
y=df["Reopened"]

#X = all others. Need to drop the answee and the ID column.
X= df.drop(columns=["Reopened","snapshot_key"])

print("Total bugs:", len(df))
print("Reopened (1): ",(y==1).sum())
print("Not Reopened (0): ",(y==0).sum())
print("Number of feature columns in X:",X.shape[1])

Total bugs: 355
Reopened (1):  46
Not Reopened (0):  309
Number of feature columns in X: 22


In [5]:
## Turning columns into numbers

# Part 2a - Numeric columns
# 12 numeric columns - since it is already numbers, just filling the blanks with 0)
NUM = ['changelog_Weekday','changelog_Month_day','changelog_Month','changelog_Day_of_year',
       'Number_of_comments','changelog_Priority_changed','Time_days','Number_of_files_changed',
       'updated_description_size','accumulated_comment_size','reporter_experience','assignee_experience']

X_num = X[NUM].fillna(0)

print("Numeric feature shape: ",X_num.shape)
X_num.head() #top 5

Numeric feature shape:  (355, 12)


,changelog_Weekday,changelog_Month_day,changelog_Month,changelog_Day_of_year,Number_of_comments,changelog_Priority_changed,Time_days,Number_of_files_changed,updated_description_size,accumulated_comment_size,reporter_experience,assignee_experience
0,3,21,8,233,0.0,0,1,0.0,39,1,4,1
1,5,23,8,235,3.0,0,0,23.0,102,46,0,0
2,6,24,8,236,3.0,0,0,5.0,64,20,1,3
3,6,24,8,236,2.0,0,0,0.0,3,4,3,14
4,1,26,8,238,5.0,0,0,0.0,103,58,2,5


In [6]:
# part 2b - Categorical columns
#Using one-hot encoding to make a separate 0/1 column for each option.

from sklearn.preprocessing import OneHotEncoder

CAT = ['changelog_Time','updated_component','updated_priority','changelog_Last_status']

#handle_unknown='ignore' - if any catg appears later
ohe = OneHotEncoder(handle_unknown='ignore')
X_cat = ohe.fit_transform(X[CAT].astype(str).fillna("missing"))

print("Categorical feature shape: ",X_cat.shape)
print("Count on how many new columns it became from original count: ")
for col, cats in zip(CAT, ohe.categories_): #skicit-learn 
    print(f" {col}:{len(cats)} options")

Categorical feature shape:  (355, 55)
Count on how many new columns it became from original count: 
 changelog_Time:4 options
 updated_component:33 options
 updated_priority:5 options
 changelog_Last_status:13 options


In [7]:
# part 2c - Text Columns using TD-IDF

from sklearn.feature_extraction.text import TfidfVectorizer

#Combine the two text columns into 1 and fill the balnks with empty string
text_combined = (X['description_pre_processed_text'].fillna("") + " "+ X['comment_pre_processed_text'].fillna(""))

tfidf = TfidfVectorizer(max_features=300) #capping it up with top 300 
X_text = tfidf.fit_transform(text_combined)
print("Text feature shape: ",X_text.shape)
print("Few example words picked: ",tfidf.get_feature_names_out()[:15]) #top 15

Text feature shape:  (355, 300)
Few example words picked:  ['abl' 'access' 'activ' 'actual' 'ad' 'add' 'address' 'alreadi' 'also'
 'anoth' 'apach' 'app' 'appear' 'appli' 'applic']


In [8]:
# Here stemming is already appliead by the author already and hence the colms are called as pre_processed text just so we don't repeat wordds with same concept.

In [9]:
# Part 2d - Combining all 3 into 1 feature table.
# X_num, X_cat and X_text
# Since X_cat and X_text are sparse matrices, we are using hstack to keep everything in sparse format. 

from scipy.sparse import hstack, csr_matrix

#Join side by side- num +catg + text
X_all = hstack([csr_matrix(X_num.values), X_cat, X_text])

print("Final feature table shape:",X_all.shape)
print("Adding 12 numeric +55 Categorical + 300 text = ",12+55+300)

Final feature table shape: (355, 367)
Adding 12 numeric +55 Categorical + 300 text =  367


In [10]:
#Part 3 - Training a model to get GERONIMO's AUC

# 1. Train/test split
# 2. SMOTE - to fix imbalance on the train pile only
# 3. Random Forest model - 
# 4. AUC score

from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

#1. Splitting into train (75%) and test (25%) with same ratio - stratify

X_tr, X_te, y_tr, y_te = train_test_split(X_all, y, test_size=0.25,
                                          stratify=y, random_state=42)

#2. Balancing the training set alone with SMOTE.
sm = SMOTE(random_state=42, k_neighbors=min(5,(y_tr ==1).sum() -1))
X_tr_bal,y_tr_bal = sm.fit_resample(X_tr, y_tr)

# 3. Training Random forest on the balanced training set
rf = RandomForestClassifier (n_estimators=100,random_state=42,n_jobs=-1)
rf.fit(X_tr_bal, y_tr_bal)

#4. Predict probability on the test set and score AUC
probs = rf.predict_proba(X_te)[:,1]
auc = roc_auc_score(y_te,probs)

print("Training set before SMOTE:", (y_tr == 1).sum(), "reopened /", len(y_tr), "total")
print("Training set after SMOTE: ", (y_tr_bal == 1).sum(), "reopened /", len(y_tr_bal), "total")
print("Test set:", (y_te == 1).sum(), "reopened /", len(y_te), "total")
print("GERONIMO AUC:", round(auc, 3))

Training set before SMOTE: 34 reopened / 266 total
Training set after SMOTE:  232 reopened / 464 total
Test set: 12 reopened / 89 total
GERONIMO AUC: 0.867


In [12]:
## Part 3b - Cross validation

from sklearn.model_selection import StratifiedKFold
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_aucs=[]

for fold_num, (tr_idx, te_idx) in enumerate(skf.split(X_all, y), start=1):
    X_tr, X_te = X_all[tr_idx], X_all[te_idx]
    y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

    #SMOTE on this fold's train only
    sm = SMOTE(random_state=42, k_neighbors=min(5, (y_tr == 1).sum() - 1))
    X_tr_bal, y_tr_bal = sm.fit_resample(X_tr, y_tr)

    rf = RandomForestClassifier (n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X_tr_bal,y_tr_bal)

    probs = rf.predict_proba(X_te)[:,1]
    auc= roc_auc_score(y_te,probs)
    fold_aucs.append(auc)
    print(f"Fold {fold_num} AUC: {round(auc, 3)}")

print("\nGERONIMO mean AUC:", round(np.mean(fold_aucs), 3),
      "±", round(np.std(fold_aucs), 3))

Fold 1 AUC: 0.774
Fold 2 AUC: 0.909
Fold 3 AUC: 0.849
Fold 4 AUC: 0.574
Fold 5 AUC: 0.843

GERONIMO mean AUC: 0.79 ± 0.116


In [19]:
## Part 4 - Apply to all 47 projects
## Build Features(X)
## evaluate(X_all,y)

import glob, os, numpy as np, pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LinearRegression
from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

NUM = ['changelog_Weekday','changelog_Month_day','changelog_Month','changelog_Day_of_year',
       'Number_of_comments','changelog_Priority_changed','Time_days','Number_of_files_changed',
       'updated_description_size','accumulated_comment_size','reporter_experience','assignee_experience']
CAT = ['changelog_Time','updated_component','updated_priority','changelog_Last_status']

def autospearman(dfnum, corr_th=0.7, vif_th=5.0):
    """Tagra-style feature declutter: drop redundant numeric features.
       Pass 1 = remove highly-correlated pairs; Pass 2 = remove high-VIF."""
    X = dfnum.fillna(0).astype(float)
    cols = list(X.columns)
    # Pass 1: correlation
    corr = X.corr(method="spearman").abs()
    changed = True
    while changed and len(cols) > 1:
        changed = False
        pair = (corr_th, None, None)
        for i in range(len(cols)):
            for j in range(i + 1, len(cols)):
                r = corr.loc[cols[i], cols[j]]
                if r > pair[0]:
                    pair = (r, cols[i], cols[j])
        if pair[1] is not None:
            a, b = pair[1], pair[2]
            ma = corr.loc[a, [c for c in cols if c != a]].mean()
            mb = corr.loc[b, [c for c in cols if c != b]].mean()
            cols.remove(a if ma >= mb else b)   # drop the one more correlated with the rest
            changed = True
    # Pass 2: VIF
    while len(cols) > 1:
        vifs = {}
        for c in cols:
            others = [o for o in cols if o != c]
            r2 = LinearRegression().fit(X[others], X[c]).score(X[others], X[c])
            vifs[c] = np.inf if r2 >= 1 else 1.0 / (1.0 - r2)
        worst = max(vifs, key=vifs.get)
        if vifs[worst] >= vif_th:
            cols.remove(worst)
        else:
            break
    return cols

def nb_text_feature(train_text, test_text, ytr):
    tfidf = TfidfVectorizer(max_features=300)
    Xtr = tfidf.fit_transform(train_text)
    Xte = tfidf.transform(test_text)
    nb = MultinomialNB().fit(Xtr, ytr)
    return nb.predict_proba(Xtr)[:, 1], nb.predict_proba(Xte)[:, 1]

def evaluate(X, y, num_keep):
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    aucs = []
    for tr, te in skf.split(X, y):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        ytr, yte = y.iloc[tr], y.iloc[te]

        Xtr_num, Xte_num = Xtr[num_keep].fillna(0), Xte[num_keep].fillna(0)

        ohe = OneHotEncoder(handle_unknown='ignore')
        Xtr_cat = ohe.fit_transform(Xtr[CAT].astype(str).fillna("missing"))
        Xte_cat = ohe.transform(Xte[CAT].astype(str).fillna("missing"))

        d_tr, d_te = nb_text_feature(Xtr['description_pre_processed_text'].fillna(""),
                                     Xte['description_pre_processed_text'].fillna(""), ytr)
        c_tr, c_te = nb_text_feature(Xtr['comment_pre_processed_text'].fillna(""),
                                     Xte['comment_pre_processed_text'].fillna(""), ytr)
        txt_tr = np.column_stack([d_tr, c_tr]); txt_te = np.column_stack([d_te, c_te])

        Xtr_all = hstack([csr_matrix(Xtr_num.values), Xtr_cat, csr_matrix(txt_tr)]).tocsr()
        Xte_all = hstack([csr_matrix(Xte_num.values), Xte_cat, csr_matrix(txt_te)]).tocsr()

        sm = SMOTE(random_state=42, k_neighbors=min(5, (ytr == 1).sum() - 1))
        Xtr_bal, ytr_bal = sm.fit_resample(Xtr_all, ytr)
        rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
        rf.fit(Xtr_bal, ytr_bal)
        aucs.append(roc_auc_score(yte, rf.predict_proba(Xte_all)[:, 1]))
    return np.mean(aucs), np.std(aucs)

DATA_DIR = r"C:\Users\ramku\OneDrive\Documents\Research Practicum\Execution\Reopened bugs dataset"
files = sorted(glob.glob(os.path.join(DATA_DIR, "joined_project_dataframe_*.csv")))

results = []
for i, f in enumerate(files, 1):
    name = os.path.basename(f).replace("joined_project_dataframe_", "").replace(".csv", "")
    df = pd.read_csv(f, low_memory=False)
    y = df["Reopened"]
    n_re = int((y == 1).sum())
    if n_re < 10:
        print(f"[{i}/47] SKIP {name}: only {n_re} reopened")
        continue
    X = df.drop(columns=["Reopened", "snapshot_key"])
    num_keep = autospearman(df[NUM])          # declutter numeric features for this project
    mean_auc, std_auc = evaluate(X, y, num_keep)
    results.append({"project": name, "n_bugs": len(df), "n_reopened": n_re,
                    "kept_numeric": len(num_keep), "mean_auc": round(mean_auc, 3), "std_auc": round(std_auc, 3)})
    print(f"[{i}/47] {name}: AUC {mean_auc:.3f} ± {std_auc:.3f}  (reopened {n_re}/{len(df)}, kept {len(num_keep)}/12 numeric)")

res = pd.DataFrame(results).sort_values("mean_auc", ascending=False)
res.to_csv(os.path.join(DATA_DIR, "baseline_results.csv"), index=False)
print("\n==== BASELINE SUMMARY ====")
print("Projects evaluated:", len(res), "| skipped:", 47 - len(res))
print("Projects with AUC >= 0.7:", int((res['mean_auc'] >= 0.7).sum()),
      f"({100 * (res['mean_auc'] >= 0.7).mean():.0f}%)")
print("Average AUC across projects:", round(res['mean_auc'].mean(), 3))

[1/47] ACCUMULO: AUC 0.645 ± 0.048  (reopened 183/2086, kept 9/12 numeric)
[2/47] AIRAVATA: AUC 0.777 ± 0.070  (reopened 107/1329, kept 10/12 numeric)
[3/47] AMBARI: AUC 0.693 ± 0.024  (reopened 835/16600, kept 9/12 numeric)
[4/47] ARIES: AUC 0.740 ± 0.117  (reopened 58/1001, kept 9/12 numeric)
[5/47] ARROW: AUC 0.730 ± 0.091  (reopened 98/2959, kept 9/12 numeric)
[6/47] ASTERIXDB: AUC 0.928 ± 0.068  (reopened 47/1665, kept 10/12 numeric)
[7/47] ATLAS: AUC 0.668 ± 0.054  (reopened 63/1913, kept 9/12 numeric)
[8/47] SKIP BAHIR: only 5 reopened
[9/47] BEAM: AUC 0.988 ± 0.018  (reopened 106/3098, kept 10/12 numeric)
[10/47] BIGTOP: AUC 0.613 ± 0.098  (reopened 47/1563, kept 9/12 numeric)
[11/47] CALCITE: AUC 0.779 ± 0.069  (reopened 87/2037, kept 9/12 numeric)
[12/47] CAMEL: AUC 0.701 ± 0.041  (reopened 276/4539, kept 9/12 numeric)
[13/47] CASSANDRA: AUC 0.998 ± 0.002  (reopened 568/7885, kept 10/12 numeric)
[14/47] COCOON: AUC 0.645 ± 0.121  (reopened 55/705, kept 10/12 numeric)
[15/47] 

In [20]:
## Part 4 - Semantic Embedding
## just a tiny demo

!pip install sentence-transformers

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')   # downloads ~90MB the first time

examples = [
    "application crashes on startup",
    "the app fails when launching",      # same meaning, totally different words
    "please update the documentation"    # unrelated meaning
]
emb = model.encode(examples)
print("Each sentence becomes a vector of length:", emb.shape[1])   # expect 384

sim = cosine_similarity(emb)   # 1.0 = identical meaning, ~0 = unrelated
print("crash  vs  fails-launching :", round(float(sim[0, 1]), 2))   # expect HIGH (~0.6-0.8)
print("crash  vs  documentation   :", round(float(sim[0, 2]), 2))   # expect LOW (~0.1)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\ramku\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ramku\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Each sentence becomes a vector of length: 384
crash  vs  fails-launching : 0.66
crash  vs  documentation   : -0.03


In [21]:
"""##Part 4b - Just for one project - Geronimo

import pandas as pd, numpy as np, os, time

DATA_DIR = r"C:\Users\ramku\OneDrive\Documents\Research Practicum\Execution\Reopened bugs dataset"
df = pd.read_csv(os.path.join(DATA_DIR, "joined_project_dataframe_GERONIMO.csv"), low_memory=False)

# RAW text for embeddings (natural language), NOT the stemmed pre_processed columns
text = (df["updated_description"].fillna("").astype(str) + ". " +
        df["accumulated_comment_text"].fillna("").astype(str)).tolist()

t0 = time.time()
emb = model.encode(text, batch_size=64, show_progress_bar=True)
print("Embeddings shape:", emb.shape)                 # expect (355, 384)
print("Time taken:", round(time.time() - t0, 1), "sec")

np.save(os.path.join(DATA_DIR, "emb_GERONIMO.npy"), emb)   # save so we never recompute
print("Saved emb_GERONIMO.npy")"""

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Embeddings shape: (355, 384)
Time taken: 42.8 sec
Saved emb_GERONIMO.npy


In [22]:
"""import numpy as np, os

df = pd.read_csv(os.path.join(DATA_DIR, "joined_project_dataframe_GERONIMO.csv"), low_memory=False)

# re-embed WITH the 2000-char truncation
text_trunc = (df["updated_description"].fillna("").astype(str) + ". " +
              df["accumulated_comment_text"].fillna("").astype(str)).str.slice(0, 2000).tolist()
emb_trunc = model.encode(text_trunc, batch_size=64, show_progress_bar=True)

# load the one we saved earlier (no truncation)
emb_saved = np.load(os.path.join(DATA_DIR, "emb_GERONIMO.npy"))

# compare them
max_diff = float(np.abs(emb_trunc - emb_saved).max())
n_changed = int((np.abs(emb_trunc - emb_saved).max(axis=1) > 1e-5).sum())
print("Shapes:", emb_trunc.shape, "vs", emb_saved.shape)
print("Biggest difference between any two numbers:", max_diff)
print("Bugs whose vector changed at all:", n_changed, "out of", len(df))"""

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Shapes: (355, 384) vs (355, 384)
Biggest difference between any two numbers: 0.0
Bugs whose vector changed at all: 0 out of 355


In [23]:
import numpy as np, os, time

df = pd.read_csv(os.path.join(DATA_DIR, "joined_project_dataframe_AMBARI.csv"), low_memory=False)

# FULL text, no truncation
text_full = (df["updated_description"].fillna("").astype(str) + ". " +
             df["accumulated_comment_text"].fillna("").astype(str)).tolist()
t0 = time.time()
emb_full = model.encode(text_full, batch_size=64, show_progress_bar=True)
print("Shape:", emb_full.shape, " Time:", round(time.time() - t0), "sec")
np.save(os.path.join(DATA_DIR, "emb_AMBARI.npy"), emb_full)
print("Saved emb_AMBARI.npy (no truncation)")

Batches:   0%|          | 0/260 [00:00<?, ?it/s]

Shape: (16600, 384)  Time: 2146 sec
Saved emb_AMBARI.npy (no truncation)


In [25]:
text_trunc = (df["updated_description"].fillna("").astype(str) + ". " +
              df["accumulated_comment_text"].fillna("").astype(str)).str.slice(0, 2000).tolist()
emb_trunc = model.encode(text_trunc, batch_size=64, show_progress_bar=True)

emb_saved = np.load(os.path.join(DATA_DIR, "emb_AMBARI.npy"))   # the full (no-truncation) one
max_diff  = float(np.abs(emb_trunc - emb_saved).max())
n_changed = int((np.abs(emb_trunc - emb_saved).max(axis=1) > 1e-5).sum())
print("Shapes:", emb_trunc.shape, "vs", emb_saved.shape)
print("Biggest difference between any two numbers:", max_diff)
print("Bugs whose vector changed at all:", n_changed, "out of", len(df))

Batches:   0%|          | 0/260 [00:00<?, ?it/s]

Shapes: (16600, 384) vs (16600, 384)
Biggest difference between any two numbers: 0.0
Bugs whose vector changed at all: 0 out of 16600


In [26]:
import glob, time, os, numpy as np, pandas as pd

files = sorted(glob.glob(os.path.join(DATA_DIR, "joined_project_dataframe_*.csv")))

for i, f in enumerate(files, 1):
    name = os.path.basename(f).replace("joined_project_dataframe_", "").replace(".csv", "")
    out = os.path.join(DATA_DIR, f"emb_{name}.npy")
    if os.path.exists(out):                       # resume: skip already-done projects
        print(f"[{i}/47] {name}: already done, skipping")
        continue
    df = pd.read_csv(f, low_memory=False)
    text = (df["updated_description"].fillna("").astype(str) + ". " +
            df["accumulated_comment_text"].fillna("").astype(str)).str.slice(0, 2000).tolist()  # truncate
    t0 = time.time()
    emb = model.encode(text, batch_size=64, show_progress_bar=False)
    np.save(out, emb)
    print(f"[{i}/47] {name}: {emb.shape}  ({round(time.time() - t0)}s)")

print("All embeddings saved.")

[1/47] ACCUMULO: (2086, 384)  (116s)
[2/47] AIRAVATA: (1329, 384)  (106s)
[3/47] AMBARI: already done, skipping
[4/47] ARIES: (1001, 384)  (65s)
[5/47] ARROW: (2959, 384)  (277s)
[6/47] ASTERIXDB: (1665, 384)  (169s)
[7/47] ATLAS: (1913, 384)  (180s)
[8/47] BAHIR: (61, 384)  (7s)
[9/47] BEAM: (3098, 384)  (263s)
[10/47] BIGTOP: (1563, 384)  (114s)
[11/47] CALCITE: (2037, 384)  (193s)
[12/47] CAMEL: (4539, 384)  (505s)
[15/47] COUCHDB: (1878, 384)  (138s)
[16/47] CTAKES: (162, 384)  (13s)
[17/47] CXF: (5002, 384)  (328s)
[18/47] FELIX: (3097, 384)  (247s)
[19/47] FINERACT: (381, 384)  (29s)
[20/47] FLEX: (27586, 384)  (2963s)
[21/47] FLINK: (6464, 384)  (777s)
[22/47] GEODE: (3854, 384)  (494s)
[23/47] GERONIMO: already done, skipping
[24/47] HADOOP: (6617, 384)  (807s)
[25/47] HAWQ: (756, 384)  (88s)
[26/47] IGNITE: (4404, 384)  (490s)
[27/47] IMPALA: (4931, 384)  (633s)
[28/47] JAMES: (1117, 384)  (113s)
[29/47] KAFKA: (4339, 384)  (521s)
[30/47] KYLIN: (2096, 384)  (198s)
[31/47] MES

In [27]:
import glob, os

emb = {os.path.basename(x).replace("emb_", "").replace(".npy", "")
       for x in glob.glob(os.path.join(DATA_DIR, "emb_*.npy"))}
csv = {os.path.basename(x).replace("joined_project_dataframe_", "").replace(".csv", "")
       for x in glob.glob(os.path.join(DATA_DIR, "joined_project_dataframe_*.csv"))}

print("Embedding files found:", len(emb))
print("Projects total:", len(csv))
missing = csv - emb
print("Missing:", sorted(missing) if missing else "none — all 47 present!")

Embedding files found: 47
Projects total: 47
Missing: none — all 47 present!


In [2]:
#Part A of Cross validation
DATA_DIR = r"C:\Users\ramku\OneDrive\Documents\Research Practicum\Execution\Reopened bugs dataset"
import glob, os, numpy as np, pandas as pd
NUM = ['changelog_Weekday','changelog_Month_day','changelog_Month','changelog_Day_of_year',
       'Number_of_comments','changelog_Priority_changed','Time_days','Number_of_files_changed',
       'updated_description_size','accumulated_comment_size','reporter_experience','assignee_experience']
files = sorted(glob.glob(os.path.join(DATA_DIR,"joined_project_dataframe_*.csv")))
num_list,emb_list,y_list,proj_list = [],[],[],[]
for f in files:
    name=os.path.basename(f).replace("joined_project_dataframe_","").replace(".csv","")
    df = pd.read_csv(f,low_memory=False)
    emb = np.load(os.path.join(DATA_DIR, f"emb_{name}.npy"))
    assert len(df) == len(emb),f"{name}: rows {len(df)} vs emb {len(emb)}" #safety: must line up
    num_list.append(df[NUM].fillna(0).astype(float).values)
    emb_list.append(emb)
    y_list.append(df["Reopened"].values)
    proj_list.append(np.array([name]* len(df)))

    X_num_all = np.vstack(num_list) # Numeric features for all bugs
    X_emb_all = np.vstack(emb_list) #embeddings for all bugs
    y_all = np.concatenate(y_list) #labels for all bugs
    proj_all = np.concatenate(proj_list) #which project each bug belongs to

print("Numeric block :", X_num_all.shape)
print("Embedding block:", X_emb_all.shape)
print("Labels        :", y_all.shape, " reopened:", int(y_all.sum()))
print("Projects       :", len(np.unique(proj_all)))

Numeric block : (168881, 12)
Embedding block: (168881, 384)
Labels        : (168881,)  reopened: 8886
Projects       : 47


In [3]:
#Part b of Cross validation 

import numpy as np, pandas as pd, os
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# numeric + embeddings, as one matrix (float32 to save memory)
X_all = np.hstack([X_num_all, X_emb_all]).astype(np.float32)
print("Full feature matrix:", X_all.shape)

projects = [p for p in np.unique(proj_all) if (y_all[proj_all == p] == 1).sum() >= 10]   # same 45 as baseline

out_csv = os.path.join(DATA_DIR, "cross_project_results.csv")
if os.path.exists(out_csv):                              # resume if interrupted
    prev = pd.read_csv(out_csv); done = set(prev["project"]); cp_results = prev.to_dict("records")
else:
    done = set(); cp_results = []

for i, p in enumerate(projects, 1):
    if p in done:
        print(f"[{i}/{len(projects)}] {p}: already done, skipping"); continue
    te = proj_all == p          # test  = the held-out project
    tr = ~te                    # train = all the OTHER projects
    rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1,
                                class_weight='balanced').fit(X_all[tr], y_all[tr])
    auc = roc_auc_score(y_all[te], rf.predict_proba(X_all[te])[:, 1])
    cp_results.append({"project": p, "n_reopened": int((y_all[te] == 1).sum()), "cp_auc": round(auc, 3)})
    pd.DataFrame(cp_results).to_csv(out_csv, index=False)        # save after each (resumable)
    print(f"[{i}/{len(projects)}] {p}: cross-project AUC {auc:.3f}")

cp = pd.DataFrame(cp_results)
print("\n==== CROSS-PROJECT SUMMARY ====")
print("Mean cross-project AUC:", round(cp['cp_auc'].mean(), 3))
print("Projects with AUC >= 0.7:", int((cp['cp_auc'] >= 0.7).sum()),
      f"({100 * (cp['cp_auc'] >= 0.7).mean():.0f}%)")

Full feature matrix: (168881, 396)
[1/45] ACCUMULO: cross-project AUC 0.557
[2/45] AIRAVATA: cross-project AUC 0.549
[3/45] AMBARI: cross-project AUC 0.528
[4/45] ARIES: cross-project AUC 0.589
[5/45] ARROW: cross-project AUC 0.606
[6/45] ASTERIXDB: cross-project AUC 0.540
[7/45] ATLAS: cross-project AUC 0.642
[8/45] BEAM: cross-project AUC 0.567
[10/45] CALCITE: cross-project AUC 0.576
[11/45] CAMEL: cross-project AUC 0.614
[12/45] CASSANDRA: cross-project AUC 0.566
[13/45] COCOON: cross-project AUC 0.630
[14/45] COUCHDB: cross-project AUC 0.628
[15/45] CXF: cross-project AUC 0.644
[16/45] FELIX: cross-project AUC 0.593
[17/45] FINERACT: cross-project AUC 0.506
[25/45] IMPALA: cross-project AUC 0.574
[26/45] JAMES: cross-project AUC 0.568
[27/45] KAFKA: cross-project AUC 0.581
[28/45] KYLIN: cross-project AUC 0.591
[29/45] MESOS: cross-project AUC 0.520
[30/45] MNG: cross-project AUC 0.598
[31/45] MYFACES: cross-project AUC 0.575
[32/45] NETBEANS: cross-project AUC 0.584
[33/45] NIFI:

In [4]:
## Cross-project validation for the 2 skipped projects.
#  where the per-project baseline couldn't evaluate.
for p in ["BAHIR", "CTAKES"]:
    te = proj_all == p; tr = ~te
    rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=4,
                                class_weight='balanced').fit(X_all[tr], y_all[tr])
    auc = roc_auc_score(y_all[te], rf.predict_proba(X_all[te])[:, 1])
    print(f"{p}: cross-project AUC {auc:.3f}  "
          f"(reopened {int((y_all[te]==1).sum())}/{int((proj_all==p).sum())})")

BAHIR: cross-project AUC 0.586  (reopened 5/61)
CTAKES: cross-project AUC 0.697  (reopened 8/162)


In [5]:
#numeric-only vs embeddings diagnostic
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(42)
for p in ["AIRAVATA", "CASSANDRA", "FLINK", "ARIES", "OODT"]:   # mix of sizes
    te = proj_all == p
    tr_idx = np.where(~te)[0]
    sub = rng.choice(tr_idx, size=min(40000, len(tr_idx)), replace=False)   # subsample train for speed
    out = {}
    for label, M in [("numeric", X_num_all), ("embeddings", X_emb_all),
                     ("both", np.hstack([X_num_all, X_emb_all]))]:
        rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=4,
                                    class_weight='balanced').fit(M[sub], y_all[sub])
        out[label] = roc_auc_score(y_all[te], rf.predict_proba(M[te])[:, 1])
    print(f"{p:11} numeric={out['numeric']:.3f}  embeddings={out['embeddings']:.3f}  both={out['both']:.3f}")

AIRAVATA    numeric=0.571  embeddings=0.500  both=0.514
CASSANDRA   numeric=0.681  embeddings=0.525  both=0.541
FLINK       numeric=0.544  embeddings=0.515  both=0.551
ARIES       numeric=0.587  embeddings=0.562  both=0.610
OODT        numeric=0.573  embeddings=0.656  both=0.605


In [ ]:
## The above tells us whether dropping the numeric features rescues cross-project or whether it is a true negative.
""" From this output, we underrstand that Embebbing is dragging the value down. They aren't just unhelpful across projects, they're
adding noise that dilutes the weak signal the numeric features have.

Hence, semantic embeddings do not transfer the reopen signal across projects.

This answers the SQ2 - do semantic embeddings improve cross-project prediction - no they don't and makes it worse as well."""

In [ ]:
""" We also used SMOTE on the baseline and class_weight on the cross project as cloning 90* 160000 row dataset would crash the device.
But, this anyway didn't change any results. The imbalance method didn't affect the findings. """

In [6]:
#Now we will test numeric-only cross project for all 47 projects.
""" From the 5 project diagnosis, we got to know that numeric-only will come out slightly better than
the embeddings version. Applying on all 47 would give a complete evidence saying cross-project 
doesn't beat per-project. """

import numpy as np, pandas as pd, os
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

projects_all = sorted(np.unique(proj_all))   # all 47

out_csv = os.path.join(DATA_DIR, "cross_project_numeric_results.csv")
if os.path.exists(out_csv):
    prev = pd.read_csv(out_csv); done = set(prev["project"]); rows = prev.to_dict("records")
else:
    done = set(); rows = []

for i, p in enumerate(projects_all, 1):
    if p in done:
        print(f"[{i}/47] {p}: already done, skipping"); continue
    te = proj_all == p; tr = ~te
    rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=4,
                                class_weight='balanced').fit(X_num_all[tr], y_all[tr])
    auc = roc_auc_score(y_all[te], rf.predict_proba(X_num_all[te])[:, 1])
    rows.append({"project": p, "n_reopened": int((y_all[te] == 1).sum()), "num_auc": round(auc, 3)})
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    print(f"[{i}/47] {p}: numeric-only AUC {auc:.3f}  (reopened {int((y_all[te]==1).sum())})")

num = pd.DataFrame(rows)
paired = num[num["n_reopened"] >= 10]        # the 45 with baselines
print("\n==== NUMERIC-ONLY CROSS-PROJECT SUMMARY ====")
print("Projects (>=10 reopened):", len(paired))
print("Mean numeric-only AUC:", round(paired['num_auc'].mean(), 3))
print("Projects with AUC >= 0.7:", int((paired['num_auc'] >= 0.7).sum()),
      f"({100 * (paired['num_auc'] >= 0.7).mean():.0f}%)")


[1/47] ACCUMULO: numeric-only AUC 0.582  (reopened 183)
[2/47] AIRAVATA: numeric-only AUC 0.608  (reopened 107)
[3/47] AMBARI: numeric-only AUC 0.541  (reopened 835)
[4/47] ARIES: numeric-only AUC 0.672  (reopened 58)
[5/47] ARROW: numeric-only AUC 0.673  (reopened 98)
[6/47] ASTERIXDB: numeric-only AUC 0.638  (reopened 47)
[7/47] ATLAS: numeric-only AUC 0.608  (reopened 63)
[8/47] BAHIR: numeric-only AUC 0.525  (reopened 5)
[9/47] BEAM: numeric-only AUC 0.742  (reopened 106)
[10/47] BIGTOP: numeric-only AUC 0.506  (reopened 47)
[11/47] CALCITE: numeric-only AUC 0.607  (reopened 87)
[12/47] CAMEL: numeric-only AUC 0.646  (reopened 276)
[13/47] CASSANDRA: numeric-only AUC 0.698  (reopened 568)
[14/47] COCOON: numeric-only AUC 0.626  (reopened 55)
[15/47] COUCHDB: numeric-only AUC 0.679  (reopened 87)
[16/47] CTAKES: numeric-only AUC 0.522  (reopened 8)
[17/47] CXF: numeric-only AUC 0.686  (reopened 228)
[18/47] FELIX: numeric-only AUC 0.607  (reopened 198)
[19/47] FINERACT: numeric-only

In [7]:
" Comparison by merging all 3 into 1 table to analyse the results"
import pandas as pd, os

b = pd.read_csv(os.path.join(DATA_DIR, "baseline_results.csv"))[["project","mean_auc"]].rename(columns={"mean_auc":"baseline"})
e = pd.read_csv(os.path.join(DATA_DIR, "cross_project_results.csv"))[["project","cp_auc"]].rename(columns={"cp_auc":"cp_embed"})
n = pd.read_csv(os.path.join(DATA_DIR, "cross_project_numeric_results.csv"))[["project","n_reopened","num_auc"]].rename(columns={"num_auc":"cp_numeric"})

m = b.merge(e, on="project").merge(n, on="project").sort_values("n_reopened")
m["delta (best CP - baseline)"] = (m["cp_numeric"] - m["baseline"]).round(3)

print(m.to_string(index=False))
print("\n=== SUMMARY (45 paired projects) ===")
for col in ["baseline","cp_embed","cp_numeric"]:
    print(f"  {col:11} mean={m[col].mean():.3f}   >=0.7: {int((m[col]>=0.7).sum())}/{len(m)}")
print("  Projects where best cross-project BEAT baseline:",
      int((m['cp_numeric'] > m['baseline']).sum()), "/", len(m))
m.to_csv(os.path.join(DATA_DIR, "comparison_all.csv"), index=False)
print("\nSaved comparison_all.csv")

  project  baseline  cp_embed  n_reopened  cp_numeric  delta (best CP - baseline)
      ORC     0.685     0.579          12       0.525                      -0.160
  PARQUET     0.717     0.607          16       0.601                      -0.116
     OODT     0.602     0.556          17       0.599                      -0.003
   SENTRY     0.627     0.590          22       0.516                      -0.111
 GERONIMO     0.799     0.532          46       0.421                      -0.378
   BIGTOP     0.613     0.555          47       0.506                      -0.107
ASTERIXDB     0.928     0.540          47       0.638                      -0.290
 FINERACT     0.728     0.506          50       0.561                      -0.167
    JAMES     0.755     0.568          52       0.600                      -0.155
 NETBEANS     0.733     0.584          54       0.664                      -0.069
   COCOON     0.645     0.630          55       0.626                      -0.019
      TEZ     0.